# Drought 2026 — Swiss grassland and water

Analysis notebook. All logic lives in `src/drought/`; this file calls it.
Settings are in `config.yaml` — change them there, not here, so a figure can
always be traced to the settings that made it.

```
config.yaml          all settings
src/drought/         modules
data/gee/<site>/     GEE exports — put the CSVs here by hand
data/labels/         hand labels, one file per site
data/masks/          grassland mask GeoTIFF (uploaded to GEE)
output/figures/<site>/
output/tables/       provenance JSON beside every result
gee/                 the Earth Engine scripts
```

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.insert(0, "../src")

import drought as dr

# What data is actually present? Say so now rather than failing five cells later.
for s in vars(dr.CFG.sites):
    print(dr.available(s))

## 1. Setup

Detection settings are FROZEN: tuned on 42 hand-labelled parcel-years, then
validated on all 65 (F1 0.911 -> 0.912). Re-tuning on the full label set would
forfeit that out-of-sample property.

In [ ]:
for k, v in vars(dr.CFG.detect).items():
    print(f"  {k:22} {v}")

# Refuses to run if a config parameter is not actually read by the detector.
dr.evaluate.assert_wired()

## 2. Cumulative precipitation and water balance

Current year and 1991-2020 reference come from the SAME MeteoSwiss station
series. The second panel is P - ET0 (Hargreaves): an index of atmospheric
demand against supply, NOT a soil water balance.

In [ ]:
dr.precip.main()

## 3. Grassland mask

Built locally from the yearly Nutzungsflaechen shapefiles, uploaded to GEE once
as an image asset. The GEE scripts read that asset; they never see the polygons.

Keep the .tif as provenance and in case the asset needs rebuilding — nothing in
this notebook reads it.

Persistence is the point: a mask from one year says "grassland in 2025", and
the trajectory then reads a 2019 value through it. A parcel that was maize in
2019 would put an arable curve into the grassland baseline.

#### How strict should persistence be? — evidence first

The strict rule drops any parcel that was not grassland, or not the same
management category, in every year.

**Parcels are matched by area overlap, not by identity.** Outlines are redrawn
between yearly deliveries, and a field can be split into two or two merged into
one. Any method that tries to follow an individual parcel through the years
fails on that — a split moves both centroids far from the original, and no
matching tolerance recovers it. Instead, each year's grassland is dissolved into
one geometry per category, and a current parcel counts as holding that category
in year Y if at least `min_overlap` of its area falls inside it. The question
becomes about ground cover, not parcel identity.

`matching_check()` shows how sensitive the result is to that threshold — a
gentle slope means parcels are either clearly the same ground or clearly not.
`category_history()` then separates "was not grassland" from "was grassland but
a different category", which is real management change.

`min_years_same` in config.yaml relaxes the rule to "held its category in at
least k of N years".

In [ ]:
from drought import mask

# Parcels are matched between years by AREA OVERLAP, not by identity: outlines
# are redrawn, split and merged between yearly deliveries, and nothing that
# tries to follow an individual parcel through the years survives that.
mask.matching_check()          # how sensitive is the set to the overlap threshold?

hist = mask.category_history() # how many years did each parcel hold its category?

In [ ]:
from drought import mask

# Rebuild the mask. Prints the master parcel set: how many fields are dropped
# for leaving grassland, how many for changing management category, and what
# remains per category.
master = mask.main()

# What each purity threshold costs, against the polygon area of that same set.
mask.area_check(site="rheintal")

# One landscape, continuous purity, with a zoomed inset.
mask.figure_purity_map("linthebene", group="intensive")

Pixel purity per landscape, and what the threshold costs. Purity is computed
PER GROUP — the `intensive` band counts only 613/650, so a cell on a 613/611
boundary is pure "Mahd" but mixed "intensiv". The intensive numbers are
therefore stricter than the mown ones, and the mown mixed-share understates the
fragmentation relevant to the cut analysis.

An explicit shape filter is unnecessary: a long narrow parcel simply has no
cells reaching the threshold, so purity already filters on shape. The
consequence is a sampling bias — surviving parcels skew compact.

#### Why 0.9? — measured, not asserted

Run `gee/gee_purity_sensitivity.js` once per site. It returns the mean index and
pixel count per purity class per year, from which the anomaly at EVERY threshold
is reconstructed — one extraction instead of one per threshold.

The parcel set is held fixed: only parcels with at least one clean interior cell
enter, so the threshold changes which CELLS are used, never which FIELDS. Pure
cells are otherwise the interiors of large compact parcels, and a drift in the
curve could then be small fields behaving differently rather than contamination.

If the curve is flat from the strict end down to 0.9, the choice is not a lever
on the result — a stronger statement than any claim that 0.9 is optimal.

In [ ]:
dr.figures.figure_purity_sensitivity(SITE, regime="intensive")

## 4. Grassland VI trajectory and season integral

Site-mean NDMI against the baseline percentile envelope. The trajectory panel
is the trustworthy one; the integral depends on how gaps are handled.

In [ ]:
SITE = "linthebene"

# The trajectory export covers ALL sites in one file (the GEE script runs them
# together), so it may sit in data/gee/ rather than in a site folder — plot()
# looks in both.
try:
    traj = dr.trajectory.plot(site=SITE, index="NDMI")
except FileNotFoundError as e:
    print("no trajectory export yet:", e)

In [ ]:
# The other indices, for comparison. Filenames carry the index, so nothing is
# overwritten. Agreement across indices is the robustness check: if all four
# point the same way, the result does not depend on which one is shown.
for ix in ["kNDVI", "NDVI", "NDRE"]:
    dr.trajectory.plot(site=SITE, index=ix, verbose=False)

### 4.1 The mask, and how much of it we measure

The mask is built polygon-first: a MASTER PARCEL SET of fields that are
grassland in every year AND hold the same management category throughout, then
four categories by LNF code within it, then PSF-weighted purity per category on
the 10 m Sentinel grid.

Persistence is a property of the parcel set, never multiplied into the raster.
That matters: the earlier cell-level version zeroed the sensor halo outside
parcels and produced an observed area SMALLER than the polygon area at zero
threshold, which is impossible.

`area_check` reports the master set and what each threshold costs. The ratio at
>0 must exceed 1 — the PSF spreads past every boundary. Crossing 1 is not the
target; it happens where about half the retained cells are contaminated.

In [ ]:
dr.trajectory.compare_masks(site=SITE, index="NDMI")

## 5. Grassland cut detection

Cleaning FLAGS haze rather than dropping it, so the plots can show what was
excluded.

In [ ]:
df = dr.load_site(SITE)
ev = dr.detect.detect_all(df)
dr.metrics.summarise(ev)

### 5.1 The result — when cutting stopped

Last cut is the most robust cut metric: the detector needs to find ONE cut per
parcel-year, not all of them, and the bias runs the right way — the drought
year has the most images, so a late cut is easier to detect there.

Cut COUNT is not comparable between years: sensitivity rises steeply with the
number of clear observations, and drought summers are sunny.

In [ ]:
lc = dr.figures.figure_last_cut(df, SITE, regime="intensive")

In [ ]:
summary = dr.figures.figure_summary(df, SITE, regime="intensive", n_rep=10)
summary.round(3)

### 5.1b When cuts happened — the whole distribution

The DOY-190 split above had to be chosen, and any choice invites "why 190?".
This shows where the cuts actually fell instead of asking.

Left: a violin per year, all detected cuts pooled across parcels — the SHAPE is
the message, and a violin is normalised so one year having more detections does
not distort it. Right: the cumulative share of each year's cuts by day. A line
rising earlier means cutting was front-loaded, and it is a ratio at every point,
so it survives detection rates differing between years.

In [ ]:
cuts = dr.figures.figure_cut_timing(df, SITE, regime="intensive")

### 5.1c How many cuts per parcel

**Read the caveat before the figure.** Cut count is the confounded metric:
detection sensitivity rises from ~1.3 cuts at <15 clear images to ~2.9 at
35-45, and drought summers are sunny. A raw count distribution compares
observing conditions as much as mowing.

Left is the raw distribution, right the same after thinning every parcel-year to
the SAME number of observations. **The difference between the panels is the size
of the artefact** — which is why the left one must never be quoted alone.

In [ ]:
counts = dr.figures.figure_cut_count(df, SITE, regime="intensive",
                                     n_rep=10)
counts["matched"].round(2)

### 5.2 Detector validation

Extensive meadows may not legally be cut before 15 June in the valley and hill
zones, so any earlier detection there is a known error — a regulation used as
ground truth.

In [ ]:
_ = dr.figures.figure_legal_check(df, SITE)

### 5.3 Inspect individual parcels

Before believing any number: is there a red line at every place your eye sees
an abrupt drop, and nowhere else?

In [ ]:
pids = ev[(ev.regime == "intensive") &
          (ev.year == dr.CFG.project.current_year)].parcel_id.head(3)
for pid in pids:
    dr.figures.plot_parcel(df, pid, ev, site=SITE)

## 6. Reference data and validation

Hand labels are the durable asset — everything else can be rebuilt from them.
One file per site: a detector validated on Linthebene has not been validated on
Rheintal.

### 6.1 Label

Click to add at the nearest observation, click a blue marker to remove. **H**
hides the detector's proposals so you can label blind, then reveals them —
labels copied from the detector measure nothing. Place each at the TROUGH, the
first point clearly down.

Saved labels load automatically and their parcel-years are excluded, so a
second session extends the set.

In [ ]:
dr.labelling.label_widget(df, SITE, ev=ev, n=20, seed=0)

In [ ]:
# paste the dict from the widget, then:
# labels = dr.labels.check_labels(labels, df)
# dr.labels.save_labels(labels, SITE)

labels = dr.labels.load_labels(SITE)

### 6.2 Score

Overall accuracy matters less than whether it is FLAT ACROSS YEARS. The
headline compares years, so year-dependent error rates are the real threat.

In [ ]:
dr.evaluate.score(labels, df)
dr.evaluate.score_by_year(labels, df)

In [ ]:
# Where do the misses fall? Only misses that are a parcel's LAST cut can bias
# the last-cut metric.
miss = dr.evaluate.miss_positions(labels, df)
print(miss.pivot_table(index="year", columns="half", values="doy",
                       aggfunc="size").fillna(0).astype(int))
print()
print(miss.groupby("year").is_last.agg(["sum", "size"]))

#### Late misses in detail

A late miss is the only kind that can bias the last-cut metric. `gap_days` is
the distance from the missed cut to the nearest detection: large means the
detector saw nothing there; small means it fired slightly off and the matching
tolerance rejected it. Different problems, different fixes.

In [ ]:
late = dr.evaluate.late_miss_report(labels, df)

# Look at the worst ones. The question for each: is there a drop your eye sees
# that the detector did not mark?
for key in late.nlargest(3, "gap_days").key:
    pid, yr = key.split("|")
    dr.figures.plot_parcel(df, pid, ev, years=[int(yr)])

#### Tuning for late cuts specifically

Overall F1 is dominated by early cuts, because there are more of them. A
detector can sit at F1 0.91 and still be poor at exactly the cuts the last-cut
metric depends on, so tune against `late_recall`, not the aggregate.

**Read `late_prec` alongside it.** A setting that lifts late recall by inventing
late cuts is worse than the problem — a spurious late cut moves the last-cut
metric in the direction you are trying to measure.

**And check your labels first.** If the cuts you spotted by eye are not IN the
label set, the scoring shares the blind spot and tuning against it proves
nothing.

In [ ]:
# Is late recall genuinely worse than early?
print(dr.evaluate.score_by_half(labels, df).to_string())
print()
print(dr.evaluate.score_by_half(labels, df, by_year=True).to_string())

In [ ]:
# Sweep against late recall. amp_window scales the threshold to a LOCAL window
# instead of the whole season — the hypothesis being that a late cut in a drying
# canopy produces a smaller absolute drop. It had no effect on synthetic data;
# this is the real test.
dr.evaluate.tune_for(labels, df, {
    "min_drop_abs":  [0.12, 0.14, 0.16, 0.18, 0.20],
    "drop_fraction": [0.25, 0.35],
    "amp_window":    [0, 45],
}, objective="late_recall").round(3).head(12)

### 6.3 The hand-labelled result

Depends on no algorithm — only on reading the curves. Both years truncated to
the same day.

In [ ]:
piv = dr.metrics.labels_paired(labels, end=dr.detect.season_end(df))
cur = dr.CFG.project.current_year
for m, lab in [("n", "cuts"), ("first", "first cut"), ("last", "LAST cut")]:
    a, b = piv[(m, 2025)], piv[(m, cur)]
    d = b - a
    print(f"{lab:>12}: 2025 {a.median():6.1f} | {cur} {b.median():6.1f} "
          f"| change {d.median():+5.1f}  ({(d < 0).sum()}/{len(d)} lower)")

### 6.4 Record what produced these numbers

A number without its configuration is not reproducible, and this one has
changed six times.

In [ ]:
dr.write_provenance(f"cuts_{SITE}", {
    "score": dr.evaluate.score(labels, df, verbose=False),
    "n_labels": sum(len(v) for v in labels.values()),
})

### Appendix — alternative tested and rejected

A penalised piecewise-linear changepoint fit scored F1 0.871 against the
heuristic's 0.911 on the same labels, over-segmenting (32 false positives
against 12). Not adopted. Kept as the record of a measured negative result.

In [ ]:
# from drought import changepoint
# changepoint.tune_changepoint(labels, df)